[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pmarcelino/mobillity-courses/blob/main/module-3-cleaning-and-exploring/notebook-3.3-system-usage.ipynb)


# Module 3 — Detecting Outliers in System Usage (Colab walkthrough)

A hands-on companion to the lecture on **detecting and handling outliers**, run on a small, **synthetic** Barcelona-Bicing-style **system-wide usage time series** (one row per 5 minutes).

## What this notebook is for

This notebook is a **preview** of what the lecture will show you. Run it top to bottom — or just read the printed outputs — to get a feel for the work before the lecture explains it: spotting values that don't fit the pattern, deciding whether each one is a real *event* or an *error*, and handling it without distorting the rest of the data. You don't need to follow every line of code yet. The goal is to **picture what we'll be doing**, so when the lecture walks through the *why*, you already have a concrete example in your head. Every step prints a result you can check.

### How to run in Google Colab
1. `Runtime ▸ Run all`. The notebook **reads the dataset straight from this course's public GitHub repo** — nothing to upload, nothing to generate.
2. Everything below reads that CSV and prints a checkable result for each thing the lecture shows.

> ⚠️ **The data is synthetic.** Rows are fabricated to be *plausible* and to match the columns, types, quirks, and numbers used in the Module 3 lecture scripts. It is **not** real Bicing data — real Bicing snapshots are ~370 MB (too big for Colab) and no public trip-level Bicing data exists, which is why this small synthetic stand-in is hosted in the course repo and read directly above.


In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")            # headless backend (works in Colab and CI)
import matplotlib.pyplot as plt

# This course's practice data lives in its public GitHub repo. We read it straight
# from there, so the notebook runs the same in Google Colab or anywhere online —
# nothing to upload, nothing to generate.
DATA_URL = ("https://raw.githubusercontent.com/pmarcelino/mobillity-courses/main/"
            "mobillity-univ/module-3-cleaning-and-exploring/data/"
            "barcelona-bicing-synthetic/")


## Detecting and Handling Outliers

*The hook:* on a Tuesday at 3 pm, system-wide usage suddenly reads **zero**. We pick a
detection lens that fits the data's shape (IQR for a skewed series), then decide
event-vs-error per group, and verify the distribution before/after.


In [2]:
sysu = pd.read_csv(f"{DATA_URL}system-usage-sample.csv")
b = sysu["bikesInUsage"]
print("rows:", len(sysu))
print("skew (>0 = right-skewed):", round(float(b.skew()), 3))

# IQR detection — robust to the long daytime tail.
Q1, Q3 = b.quantile(0.25), b.quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
flagged = sysu[(b < lower) | (b > upper)]
print(f"Q1={Q1}  Q3={Q3}  IQR={IQR}")
print(f"IQR fences: [{lower:.1f}, {upper:.1f}]")
print("IQR-flagged rows:", len(flagged), "(routine rush hours are NOT flagged)")
print(flagged[["dateTime", "bikesInUsage"]].to_string(index=False))


rows: 5184
skew (>0 = right-skewed): 0.756
Q1=41.0  Q3=217.0  IQR=176.0
IQR fences: [-223.0, 481.0]
IQR-flagged rows: 4 (routine rush hours are NOT flagged)
           dateTime  bikesInUsage
2019-03-14 09:00:00          1500
2019-03-16 12:00:00           658
2019-03-16 12:05:00           627
2019-03-16 12:10:00           658


In [3]:
# Why IQR and not Z-score here? Z-score assumes a roughly SYMMETRIC spread.
mean, std = b.mean(), b.std()
z = (b - mean) / std
print(f"mean={mean:.1f}  std={std:.1f}")
print("Z-score (|z|>3) flags:", int((z.abs() > 3).sum()), "rows")
print("This series is right-skewed, so the mean/std are pulled by the busy daytime;")
print("IQR (quartile-based) is the appropriate lens for skewed usage data.")


mean=138.1  std=95.0
Z-score (|z|>3) flags: 4 rows
This series is right-skewed, so the mean/std are pulled by the busy daytime;
IQR (quartile-based) is the appropriate lens for skewed usage data.


In [4]:
# Domain thresholds catch values that are physically impossible.
FLEET = 900   # there are not more bikes in use than exist
impossible = sysu[(b < 0) | (b > FLEET)]
print("domain-impossible rows (negative, or > fleet of 900):")
print(impossible[["dateTime", "bikesInUsage"]].to_string(index=False))
print("Note: the negative value is BELOW the IQR lower fence's reach but caught here.")


domain-impossible rows (negative, or > fleet of 900):
           dateTime  bikesInUsage
2019-03-10 04:00:00            -4
2019-03-14 09:00:00          1500
Note: the negative value is BELOW the IQR lower fence's reach but caught here.


In [5]:
# Event or error? The timestamp gives the number meaning.
tue_3pm = sysu[sysu["dateTime"].str.startswith("2019-03-05 15:")]
print("Tuesday 2019-03-05, 15:xx (should be busy):")
print(tue_3pm[["dateTime", "bikesInUsage", "error"]].to_string(index=False))
typ_3pm = sysu[sysu["dateTime"].str.contains(" 15:00:")]["bikesInUsage"].mean()
typ_3am = sysu[sysu["dateTime"].str.contains(" 03:00:")]["bikesInUsage"].mean()
print(f"typical 15:00 usage: {typ_3pm:.0f}  -> the Tuesday zero is suspicious (likely error)")
print(f"typical 03:00 usage: {typ_3am:.0f}  -> near-zero at 3am is normal night behaviour")


Tuesday 2019-03-05, 15:xx (should be busy):
           dateTime  bikesInUsage  error
2019-03-05 15:00:00             0      1
2019-03-05 15:05:00             0      1
2019-03-05 15:10:00             0      1
2019-03-05 15:15:00             0      1
2019-03-05 15:20:00           216      0
2019-03-05 15:25:00           218      0
2019-03-05 15:30:00           243      0
2019-03-05 15:35:00           207      0
2019-03-05 15:40:00           200      0
2019-03-05 15:45:00           215      0
2019-03-05 15:50:00           215      0
2019-03-05 15:55:00           204      0
typical 15:00 usage: 172  -> the Tuesday zero is suspicious (likely error)
typical 03:00 usage: 25  -> near-zero at 3am is normal night behaviour


In [6]:
# Handle: cap an event group for typical-behaviour stats, or keep-and-flag.
capped = b.clip(lower=lower, upper=upper)
print("cap: max before =", int(b.max()), "-> after =", int(capped.max()))

sysu_flagged = sysu.copy()
sysu_flagged["is_outlier"] = ((b < lower) | (b > upper)).astype(int)
print("keep-and-flag: rows with is_outlier=1 :", int(sysu_flagged["is_outlier"].sum()))

# Verify the distribution changed the way we intended.
print("before -> skew %.3f, max %d" % (b.skew(), b.max()))
print("after  -> skew %.3f, max %d" % (capped.skew(), capped.max()))


cap: max before = 1500 -> after = 481
keep-and-flag: rows with is_outlier=1 : 4
before -> skew 0.756, max 1500
after  -> skew 0.152, max 481


In [7]:
# Visual verification (saved as PNGs).
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].hist(b, bins=40); ax[0].set_title("bikesInUsage — before"); ax[0].set_xlabel("bikes in use")
ax[1].hist(capped, bins=40); ax[1].set_title("after capping at IQR fence"); ax[1].set_xlabel("bikes in use")
fig.tight_layout(); fig.savefig("fig_3.3_distribution.png", dpi=90); plt.close(fig)
print("saved:", "fig_3.3_distribution.png")


saved: fig_3.3_distribution.png


## Done — system usage

The outliers lecture has run on the synthetic usage series: IQR fences that leave routine
rush hours alone, domain thresholds that catch the impossible values, the timestamp
context that tells an event from an error, and the cap/keep-and-flag handling with a
before/after distribution. Re-run any time — the notebook reads the same published data,
so the numbers don't change. Then try the matching exercise yourself.
